# Probe & baseline results

Reads `runs/`. No model, no GPU.

Figures come from `report.py`, so the notebook and `uv run python report.py`
produce identical charts. Use the script when you just want the PNGs; use this
when you want to poke at individual answers.

The question behind all of it: **does the model know enough chess for RL to have
anything to amplify?** RL boosts responses a model already gives sometimes. It
cannot install a missing capability.

In [ ]:
%matplotlib inline
from report import (
    fig_attempt_rate, fig_baseline_outcomes, fig_move_distribution,
    fig_move_quality, fig_probe_scores, load_baselines, load_probes,
    model_arm, print_summary, size_of,
)

probes = load_probes()
baselines = load_baselines()

print_summary(probes, baselines)

## 1. Capability probe

`moves` asks where a lone piece can go on an empty board. Does it know the rules?
`board` asks what sits on a square after some movetext. Can it track state?

Half the `board` squares are empty by construction, so **always answering
"empty" scores 50%**. At or below that line is guessing, not tracking.

In [ ]:
fig_probe_scores(probes)

A score at the baseline can mean never attempting, or attempting and failing.
Those need different fixes, and only this tells them apart.

In [ ]:
fig_attempt_rate(probes)

### The exact prompt, and what each model said

Verbatim, so you can see whether a wrong answer was close or nonsense. Change
`kind` and `index` to walk through the questions.

In [ ]:
def show(kind="moves", index=0, chars=600):
    keys = sorted(probes, key=lambda k: (size_of(k[0]), k[1]))
    ref = [i for i in probes[keys[0]] if i["kind"] == kind][index]

    print("=" * 74)
    print(ref["prompt"])
    print(f"\nCORRECT: {ref['answer']}")

    for key in keys:
        item = [i for i in probes[key] if i["kind"] == kind][index]
        print("-" * 74)
        print(f"{key[0]} {key[1]}  {'CORRECT' if item['score'] else 'wrong'}   "
              f"{item.get('tokens', 0)} tokens"
              f"{', truncated' if item.get('truncated') else ''}")
        print(f"extracted: {item['said']!r}\n")
        print(item.get("completion", "(not saved)")[:chars], "\n")


show("moves", index=0)

In [ ]:
show("board", index=0, chars=450)

## 2. Baseline runs

Four outcomes. The summary line collapsed them into "answered 95%", which was
mostly the model echoing the prompt's own `MOVE` placeholder.

In [ ]:
fig_baseline_outcomes(baselines)

## 3. Move quality and the degenerate-policy check

Cumulative, so medians read straight off 0.5. Left and up is better.

The second chart is the one that separates learning from collapse: a published
GRPO run on 8B models converged on pushing the a-pawn over 80% of the time,
while mean cp_loss improved the whole way.

In [ ]:
RUN = "movetext-nothink"

fig_move_quality(baselines[RUN], RUN)

In [ ]:
fig_move_distribution(baselines[RUN], RUN)

## 4. Read a completion

Numbers say a run failed. Only the text says why.

In [ ]:
row = model_arm(baselines[RUN])["rows"][0]
print(f"raw    {row['raw']!r}")
print(f"move   {row['move']}   legal={row['legal']}  cp_loss={row['cp_loss']}")
print(f"tokens {row['tokens']}  truncated={row['truncated']}")
print("\n--- completion ---")
print(row["completion"][:1200])